# Statistical Tests
## HBV-Related Hepatocellular Carcinoma (HBV-HCC) Early Detection

This notebook performs McNemar's test to determine whether the differences
in performance between the three trained models are statistically significant.

**Input:** Trained models from `models/`, `data/X_lasso.csv`, `data/y_labels.csv`  
**Test:** McNemar's test (pairwise comparison of classifiers)

In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from statsmodels.stats.contingency_tables import mcnemar

# Load data
X = pd.read_csv('../data/X_lasso.csv', index_col=0)
y = pd.read_csv('../data/y_labels.csv', index_col=0).squeeze()

# Recreate the same train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Load saved models
rf = joblib.load('../models/random_forest.pkl')
xgb = joblib.load('../models/xgboost.pkl')
svm = joblib.load('../models/svm.pkl')

print("Models and data loaded successfully!")
print(f"Test set size: {X_test.shape[0]} samples")

Models and data loaded successfully!
Test set size: 94 samples


## McNemar's Test
McNemar's test compares two classifiers by examining cases where one
classifier is correct and the other is wrong. A p-value below 0.05
indicates a statistically significant difference between the two models.

In [2]:
# Get predictions from each model
rf_preds = rf.predict(X_test)
xgb_preds = xgb.predict(X_test)
svm_preds = svm.predict(X_test)

def run_mcnemar(preds_a, preds_b, name_a, name_b, y_true):
    # Build contingency table
    correct_a = (preds_a == y_true)
    correct_b = (preds_b == y_true)
    
    # Contingency table
    # [both correct, a correct b wrong]
    # [a wrong b correct, both wrong]
    table = np.array([
        [(correct_a & correct_b).sum(), (correct_a & ~correct_b).sum()],
        [(~correct_a & correct_b).sum(), (~correct_a & ~correct_b).sum()]
    ])
    
    result = mcnemar(table, exact=True)
    
    print(f"\n{name_a} vs {name_b}")
    print(f"Contingency Table:\n{table}")
    print(f"p-value: {result.pvalue:.4f}")
    if result.pvalue < 0.05:
        print("Result: Statistically significant difference (p < 0.05)")
    else:
        print("Result: No statistically significant difference (p >= 0.05)")

# Run pairwise comparisons
run_mcnemar(rf_preds, xgb_preds, "Random Forest", "XGBoost", y_test)
run_mcnemar(rf_preds, svm_preds, "Random Forest", "SVM", y_test)
run_mcnemar(xgb_preds, svm_preds, "XGBoost", "SVM", y_test)


Random Forest vs XGBoost
Contingency Table:
[[91  0]
 [ 1  2]]
p-value: 1.0000
Result: No statistically significant difference (p >= 0.05)

Random Forest vs SVM
Contingency Table:
[[91  0]
 [ 1  2]]
p-value: 1.0000
Result: No statistically significant difference (p >= 0.05)

XGBoost vs SVM
Contingency Table:
[[92  0]
 [ 0  2]]
p-value: 1.0000
Result: No statistically significant difference (p >= 0.05)


## Summary

McNemar's test was applied to perform pairwise statistical comparisons
between all three trained classifiers on the held-out test set of 94 samples.

| Comparison | p-value | Significant? |
|------------|---------|--------------|
| Random Forest vs XGBoost | 1.0000 | No |
| Random Forest vs SVM | 1.0000 | No |
| XGBoost vs SVM | 1.0000 | No |

No statistically significant difference was found between any pair of models
(p >= 0.05 in all cases). This indicates that all three classifiers achieved
consistently high performance on the same test samples, suggesting that the
71 LASSO-selected gene expression features carry sufficiently strong
biological signal to be reliably exploited by multiple algorithmic approaches.

XGBoost remains the recommended model based on its highest accuracy (97.87%)
and